# nb_bronze_stream_vehicle_battery

**Source :** Azure Event Hub — `vehicle-battery-live` topic  
**Sink   :** `abfss://bronze@evdatalakedev2.dfs.core.windows.net/event-stream/vehicle_battery_live/`  
**Format :** Delta (append), partitioned by `event_date`  
**Trigger:** Every 30 seconds (micro-batch streaming)

## Connector: azure-eventhubs-spark (native EventHubs connector)

Uses the native `format("eventhubs")` connector via Maven JAR:  
`com.microsoft.azure:azure-eventhubs-spark_2.12:2.3.22`

**Cluster requirement:** Single User mode (not Shared/Serverless)  
Single User clusters skip the artifact allowlist, so the JAR installs fine.

**Cluster setup (one-time):**  
Cluster → Libraries → Install New → Maven → Coordinates:  
`com.microsoft.azure:azure-eventhubs-spark_2.12:2.3.22`

**Key Vault secret needed:** `eventhub-vehicle-battery-listen-conn-str`  
Value: the full SAS connection string for `databricks-listen-policy`  
(includes EntityPath=vehicle-battery-live)

| Column | Description |
|---|---|
| `body` | Raw JSON bytes from Event Hub |
| `enqueuedTime` | Event Hub enqueue timestamp |
| `sequenceNumber` | Partition sequence number |
| `offset` | Partition byte offset |
| `publisher` | Publisher name (if set) |
| `partitionKey` | Partition key |
| `systemProperties` | Map of system properties |

In [ ]:
# ── Cell 1: secrets ───────────────────────────────────────────────────────────
# eventhub-vehicle-battery-listen-conn-str = full SAS connection string
# for databricks-listen-policy (Listen only — do NOT use the producer key here)
EH_CONN_STR   = dbutils.secrets.get(scope="ev-kv-scope", key="eventhub-vehicle-battery-listen-conn-str")

SP_CLIENT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-id")
SP_CLIENT_SEC = dbutils.secrets.get(scope="ev-kv-scope", key="sp-client-secret")
SP_TENANT_ID  = dbutils.secrets.get(scope="ev-kv-scope", key="sp-tenant-id")
ADLS_ACCOUNT  = dbutils.secrets.get(scope="ev-kv-scope", key="adls-account-name")

print("Secrets loaded.")

In [ ]:
# ── Cell 2: ADLS OAuth config ─────────────────────────────────────────────────
spark.conf.set(f"fs.azure.account.auth.type.{ADLS_ACCOUNT}.dfs.core.windows.net",       "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{ADLS_ACCOUNT}.dfs.core.windows.net",
               "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{ADLS_ACCOUNT}.dfs.core.windows.net",     SP_CLIENT_ID)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{ADLS_ACCOUNT}.dfs.core.windows.net", SP_CLIENT_SEC)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{ADLS_ACCOUNT}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{SP_TENANT_ID}/oauth2/token")
print("ADLS OAuth configured.")

In [ ]:
# ── Cell 3: paths + EventHubs config ──────────────────────────────────────────
EVENTHUB_NAME   = "vehicle-battery-live"
CONSUMER_GROUP  = "$Default"

BRONZE_PATH     = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/event-stream/vehicle_battery_live/"
CHECKPOINT_PATH = f"abfss://bronze@{ADLS_ACCOUNT}.dfs.core.windows.net/_checkpoints/vehicle-battery-live/"

# encrypt() in azure-eventhubs-spark 2.3.22 takes only the connection string
jvm = spark.sparkContext._jvm
encrypted_conn_str = jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(EH_CONN_STR)

EH_CONF = {
    "eventhubs.connectionString": encrypted_conn_str,
    "eventhubs.consumerGroup": CONSUMER_GROUP,
    "eventhubs.maxEventsPerTrigger": 1000,
}

print(f"Bronze path    : {BRONZE_PATH}")
print(f"Checkpoint     : {CHECKPOINT_PATH}")
print(f"Event Hub topic: {EVENTHUB_NAME}")
print(f"Consumer group : {CONSUMER_GROUP}")

In [ ]:
# ── Cell 4: read stream via EventHubs connector ───────────────────────────────
# EventHubs delivers: body (binary), partition, offset, sequenceNumber,
#                     enqueuedTime, publisher, partitionKey, systemProperties
raw_stream = (
    spark.readStream
    .format("eventhubs")
    .options(**EH_CONF)
    .load()
)

print("Raw EventHubs stream schema:")
raw_stream.printSchema()

In [ ]:
# ── Cell 5: parse JSON from the body column ───────────────────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

EVENT_SCHEMA = StructType([
    StructField("event_id",                   StringType(),  True),
    StructField("vehicle_id",                 StringType(),  True),
    StructField("session_id",                 StringType(),  True),
    StructField("station_id",                 StringType(),  True),
    StructField("charger_id",                 StringType(),  True),
    StructField("battery_pct",                DoubleType(),  True),
    StructField("charging_rate_kw",           DoubleType(),  True),
    StructField("battery_temp_c",             DoubleType(),  True),
    StructField("state_of_charge_target_pct", IntegerType(), True),
    StructField("estimated_minutes_to_full",  IntegerType(), True),
    StructField("event_ts",                   StringType(),  True),
])

parsed = (
    raw_stream
    # EventHubs body is binary — cast to string first
    .withColumn("body_str", F.col("body").cast("string"))
    .withColumn("parsed",   F.from_json(F.col("body_str"), EVENT_SCHEMA))
    .select(
        F.col("parsed.*"),
        # EventHubs metadata columns
        F.col("enqueuedTime") .alias("eh_enqueued_time"),
        F.col("offset")       .alias("eh_offset"),
        F.col("sequenceNumber").alias("eh_sequence_number"),
        F.col("partition")    .alias("eh_partition"),
    )
)

print("Parsed schema:")
parsed.printSchema()

In [ ]:
# ── Cell 6: add Bronze metadata columns ───────────────────────────────────────
bronze_df = (
    parsed
    .withColumn("_ingestion_ts", F.current_timestamp())
    .withColumn("_source",       F.lit("eventhub/vehicle-battery-live"))
    # Flag rows where JSON parsing failed (body was not valid JSON)
    .withColumn("_is_corrupt",   F.col("event_id").isNull())
    # Partition column for Hive-style partitioning on ADLS
    .withColumn("event_date",    F.to_date(F.col("event_ts")))
)

print("Bronze schema:")
bronze_df.printSchema()

In [ ]:
# ── Cell 7: write stream to Bronze Delta ──────────────────────────────────────
streaming_query = (
    bronze_df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .option("mergeSchema", "true")
    .partitionBy("event_date")
    .trigger(processingTime="30 seconds")
    .start(BRONZE_PATH)
)

print(f"Stream started. Query ID: {streaming_query.id}")

In [ ]:
# ── Cell 8: keep-alive monitor loop ───────────────────────────────────────────
import time

while streaming_query.isActive:
    prog = streaming_query.lastProgress
    if prog:
        rows_in  = prog.get("numInputRows", 0)
        rows_sec = prog.get("inputRowsPerSecond", 0.0)
        batch    = prog.get("batchId", "?")
        print(f"[Batch {batch}] inputRows={rows_in}  rate={rows_sec:.1f} rows/s")
    time.sleep(30)

streaming_query.awaitTermination()

In [ ]:
# ── Cell 9: optional verify (run manually, not in job) ────────────────────────
verify_df = spark.read.format("delta").load(BRONZE_PATH)
print(f"Total Bronze rows: {verify_df.count()}")
verify_df.orderBy(F.col("_ingestion_ts").desc()).limit(5).display()